# Sprint 1 — CLIP Vektör Pipeline

**WhatsApp Akıllı Stil Asistanı** — B2B Multi-Tenant mimari

Bu notebook:
1. `photos/` klasöründen test alt kümesi oluşturur (`TEST_SUBSET_SIZE` → `src/config.py`)
2. **CLIP** ile embedding üretir (parametreler `src/config.py` → `CLIP_*`)
3. Vektörleri **ChromaDB**'ye `company_id="test_firmasi_1"` ile kaydeder
4. Eşik filtreli benzerlik araması yapar

## Hücre 1 — Ortam Kurulumu ve Importlar

In [2]:
import logging
import sys
from pathlib import Path

# Proje kökünü Python path'e ekle (src paketi için)
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    BATCH_SIZE,
    CHROMA_COLLECTION_NAME,
    CHROMA_PERSIST_DIR,
    CLIP_MODEL_ID,
    CLIP_PAD_TO_SQUARE,
    CLIP_SIMILARITY_THRESHOLD,
    CLIP_TOP_K,
    DEFAULT_COMPANY_ID,
    PHOTOS_DIR,
    RANDOM_SEED,
    TEST_MANIFEST_PATH,
    TEST_SUBSET_DIR,
    TEST_SUBSET_SIZE,
)
from src.dataset import TestSubsetBuilder
from src.models import get_clip_extractor
from src.vector_store import ChromaVectorStore

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("sprint1")

COMPANY_ID = DEFAULT_COMPANY_ID
logger.info("Proje kökü: %s | Kiracı: %s", PROJECT_ROOT, COMPANY_ID)

2026-06-30 23:27:07,867 | INFO | sprint1 | Proje kökü: /Users/mac/Desktop/Wp_Bot | Kiracı: test_firmasi_1


## Hücre 2 — 50 Fotoğraflık Test Veri Seti Oluşturma

Thumb görseller mümkün olduğunca elenir; `data/test_subset/` altına kopyalanır ve manifest yazılır.

In [3]:
try:
    builder = TestSubsetBuilder(
        source_dir=PHOTOS_DIR,
        output_dir=TEST_SUBSET_DIR,
        manifest_path=TEST_MANIFEST_PATH,
        subset_size=TEST_SUBSET_SIZE,
        seed=RANDOM_SEED,
        copy_files=True,
    )
    selected_paths = builder.build()
    print(f"✓ {len(selected_paths)} görsel seçildi")
    print(f"  Manifest: {TEST_MANIFEST_PATH}")
    print(f"  Kopya klasörü: {TEST_SUBSET_DIR}")
    print(f"\nİlk 5 dosya:")
    for p in selected_paths[:5]:
        print(f"  - {p.name}")
except Exception as exc:
    logger.exception("Test veri seti oluşturulamadı: %s", exc)
    raise

2026-06-30 23:27:07,922 | INFO | src.dataset.test_subset_builder | 3000 görsel seçildi | manifest: /Users/mac/Desktop/Wp_Bot/data/test_subset_manifest.json


✓ 3000 görsel seçildi
  Manifest: /Users/mac/Desktop/Wp_Bot/data/test_subset_manifest.json
  Kopya klasörü: /Users/mac/Desktop/Wp_Bot/data/test_subset

İlk 5 dosya:
  - 0006.01.019.00211_0166_1.jpg
  - 0006.01.019.00218_0214_1.jpg
  - 0006.01.019.00219_0004_1.jpg
  - 0006.01.019.00220_0461_1.jpg
  - 0006.01.019.00224_0134_1.jpg


## Hücre 3 — CLIP Model Başlatma

Parametreler `src/config.py` dosyasındaki `CLIP_*` sabitlerinden okunur.
Model, pad ve normalize ayarlarını oradan değiştirebilirsiniz.

In [4]:
try:
    extractor = get_clip_extractor()
    print(f"✓ CLIP yüklendi | model={CLIP_MODEL_ID}")
    print(f"  Boyut: {extractor.embedding_dim} | Pad: {CLIP_PAD_TO_SQUARE} | Cihaz: {extractor.device}")
except Exception as exc:
    logger.exception("CLIP yüklenemedi (ilk çalıştırmada ağırlıklar indirilir): %s", exc)
    raise

2026-06-30 23:27:09,272 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-06-30 23:27:09,432 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-06-30 23:27:09,594 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
2026-06-30 23:27:09,595 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-30 23:27:09,756 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-06-30 23:27:09,921 | INFO | httpx | HTTP Request: HEAD https://huggi

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

2026-06-30 23:27:12,592 | INFO | src.models.feature_extractor | clip hazır | model=openai/clip-vit-base-patch32 | boyut=512 | pad=True | cihaz=cpu


✓ CLIP yüklendi | model=openai/clip-vit-base-patch32
  Boyut: 512 | Pad: True | Cihaz: cpu


## Hücre 4 — ChromaDB Bağlantısı (Multi-Tenant)

Tüm kayıtlar `metadata.company_id` alanında kiracı etiketi taşır.

In [5]:
try:
    vector_store = ChromaVectorStore(
        persist_directory=CHROMA_PERSIST_DIR,
        collection_name=CHROMA_COLLECTION_NAME,
        company_id=COMPANY_ID,
    )
    print(f"✓ ChromaDB hazır | Persist: {CHROMA_PERSIST_DIR}")
    print(f"  Mevcut kiracı kayıt sayısı: {vector_store.count_for_tenant()}")
except Exception as exc:
    logger.exception("ChromaDB başlatılamadı: %s", exc)
    raise

2026-06-30 23:27:12,725 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/commits/main "HTTP/1.1 200 OK"
2026-06-30 23:27:12,805 | INFO | src.vector_store.chroma_store | ChromaDB koleksiyonu hazır | firma: test_firmasi_1 | kayıt: 0


✓ ChromaDB hazır | Persist: /Users/mac/Desktop/Wp_Bot/chroma_db
  Mevcut kiracı kayıt sayısı: 0


## Hücre 5 — Embedding Üretimi ve ChromaDB'ye Kayıt

- `torch.inference_mode()` ile RAM dostu çıkarım
- Batch boyutu: `BATCH_SIZE` (varsayılan 16)
- Her vektörde zorunlu `company_id` metadata

In [6]:
from tqdm.auto import tqdm

# Test alt kümesindeki dosya yollar
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
image_paths = [
    p for p in TEST_SUBSET_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
]
image_paths.sort()

if not image_paths:
    raise FileNotFoundError(
        f"Test klasörü boş: {TEST_SUBSET_DIR}. Önce Hücre 2'yi çalıştırın."
    )

print(f"İşlenecek görsel sayısı: {len(image_paths)}")

try:
    all_embeddings, all_ids = extractor.extract_from_paths(
        image_paths, batch_size=BATCH_SIZE
    )

    saved = vector_store.add_embeddings(
        ids=all_ids,
        embeddings=all_embeddings,
        extra_metadata=[{"model": "clip"}] * len(all_ids),
    )
    print(f"\n✓ {saved} vektör ChromaDB'ye kaydedildi | company_id={COMPANY_ID}")
    print(f"  Toplam kiracı kaydı: {vector_store.count_for_tenant()}")

except Exception as exc:
    logger.exception("Embedding / kayıt hatası: %s", exc)
    raise

2026-06-30 23:27:13,001 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/discussions?p=0 "HTTP/1.1 200 OK"


İşlenecek görsel sayısı: 3121


2026-06-30 23:27:13,301 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/commits/refs%2Fpr%2F66 "HTTP/1.1 200 OK"
2026-06-30 23:27:13,465 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/refs%2Fpr%2F66/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-06-30 23:27:13,624 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/refs%2Fpr%2F66/model.safetensors "HTTP/1.1 302 Found"



✓ 3121 vektör ChromaDB'ye kaydedildi | company_id=test_firmasi_1
  Toplam kiracı kaydı: 3121


## Hücre 6 — Benzerlik Araması (Eşik Filtreli)

`CLIP_SIMILARITY_THRESHOLD` altındaki sonuçlar elenir. Eşik ve top-k `src/config.py` içinden ayarlanır.

In [7]:
query_path = image_paths[0]
batch, _ = extractor.preprocess_paths([query_path])

if batch.numel() == 0:
    raise RuntimeError(f"Sorgu görseli işlenemedi: {query_path}")

query_vector = extractor.extract(batch)[0].tolist()

try:
    results = vector_store.query_similar(
        query_vector,
        n_results=CLIP_TOP_K + 1,
        min_similarity=CLIP_SIMILARITY_THRESHOLD,
    )

    print(f"Sorgu görseli: {query_path.name}")
    print(f"Kiracı filtresi: company_id={COMPANY_ID}")
    print(f"Eşik: {CLIP_SIMILARITY_THRESHOLD:.0%} benzerlik\n")
    print(f"En benzer ürünler (max {CLIP_TOP_K}):")
    print("-" * 50)

    shown = 0
    for doc_id, distance, meta in zip(
        results["ids"][0],
        results["distances"][0],
        results.get("metadatas", [[]])[0] or [{}] * len(results["ids"][0]),
    ):
        if doc_id == query_path.name:
            continue
        shown += 1
        similarity_pct = max(0.0, (1.0 - distance)) * 100
        print(f"{shown}. {doc_id}")
        print(f"   Mesafe: {distance:.4f} | Benzerlik: ~{similarity_pct:.1f}%")
        print(f"   Metadata: {meta}")
        if shown >= CLIP_TOP_K:
            break

    if shown == 0:
        print("Eşiği geçen sonuç yok — CLIP_SIMILARITY_THRESHOLD değerini düşürmeyi deneyin.")

except Exception as exc:
    logger.exception("Benzerlik araması başarısız: %s", exc)
    raise

Sorgu görseli: 0006.01.019.00211_0166_1.jpg
Kiracı filtresi: company_id=test_firmasi_1
Eşik: 70% benzerlik

En benzer ürünler (max 5):
--------------------------------------------------
1. 0006.01.019.00226_0077_1.jpg
   Mesafe: 0.0577 | Benzerlik: ~94.2%
   Metadata: {'model': 'clip', 'company_id': 'test_firmasi_1'}
2. 0055.01.019.00185_0011_1.jpg
   Mesafe: 0.0673 | Benzerlik: ~93.3%
   Metadata: {'model': 'clip', 'company_id': 'test_firmasi_1'}
3. 0220.01.027.00268_0694_1.jpg
   Mesafe: 0.0733 | Benzerlik: ~92.7%
   Metadata: {'company_id': 'test_firmasi_1', 'model': 'clip'}
4. 0220.01.027.00257_0077_1.jpg
   Mesafe: 0.0753 | Benzerlik: ~92.5%
   Metadata: {'model': 'clip', 'company_id': 'test_firmasi_1'}
5. 0220.01.027.00287_0135_1.jpg
   Mesafe: 0.0769 | Benzerlik: ~92.3%
   Metadata: {'model': 'clip', 'company_id': 'test_firmasi_1'}


## Hücre 7 — Özet ve Sonraki Adımlar

| Adım | Konum |
|------|-------|
| Test alt kümesi | `data/test_subset/` |
| CLIP parametreleri | `src/config.py` → `CLIP_*` |
| CLIP extractor | `src/models/feature_extractor.py` |
| ChromaDB + `company_id` | `chroma_db/collection_clip` |

**Özelleştirme ipuçları:**
- `CLIP_MODEL_ID` → `clip-vit-large-patch14` daha iyi kalite
- `CLIP_PAD_TO_SQUARE = True` → ürün kırpılmasını önler
- `CLIP_SIMILARITY_THRESHOLD` → düşük skorlu eşleşmeleri eler (0.65–0.80 arası deneyin)

**Sprint 2:** Tüm fotoğraflar için batch pipeline, FastAPI endpoint, WhatsApp webhook.